In [1]:
import pandas as pd

For questions 1 and 2: 
Given a dataset with time series data containing an event, use a linear regression to test whether there was a discontinuity in the data at the event. Consider the possibility, first, of a discontinuity only in the value of the variable but not the derivative. Then consider that there may be a discontinuity in the first derivative (the slope).  
Use the file homework_3.1.csv. 

In [3]:
df = pd.read_csv('homework_3.1.csv')
df

,time,value1,value2,value3
0,0,1.764052,1.883151,-0.369182
1,1,0.420157,-1.327759,-0.219379
2,2,1.018738,-1.230485,1.139660
3,3,2.300893,1.029397,0.715264
4,4,1.947558,-1.093123,0.720132
...,...,...,...,...
95,95,5.856573,3.628454,5.936891
96,96,5.230500,4.611791,4.937725
97,97,7.075870,4.703504,5.462954
98,98,5.486912,6.083236,4.520551


In [5]:
df.describe()

,time,value1,value2,value3
count,100.000000,100.000000,100.000000,100.000000
mean,49.500000,2.152308,1.807013,2.155768
std,29.011492,2.052524,1.557664,2.126580
min,0.000000,-2.152990,-1.327759,-1.516956
25%,24.750000,0.543743,0.963499,0.340994
50%,49.500000,1.948924,1.723450,1.816849
75%,74.250000,3.433771,2.796829,3.912036
max,99.000000,7.075870,6.083236,6.983917


In [6]:
df.isnull().sum()

time      0
value1    0
value2    0
value3    0
dtype: int64

In [14]:
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf

# 1. Load the dataset (replace path with your actual file location)
df = pd.read_csv('homework_3.1.csv')

# 2. Create the RDD running variables
cutoff = 50
df['time_centered'] = df['time'] - cutoff          # Centers time around the event (t=0 at cutoff)
df['post_event'] = (df['time'] >= cutoff).astype(int)  # Indicator: 1 if after/at event, 0 before
df['interaction'] = df['time_centered'] * df['post_event']  # Allows slope to change post-event

# List of dependent variables to test
value_cols = ['value1', 'value2', 'value3']

# 3. Run RDD for each dataset
for col in value_cols:
    print(f"\n" + "="*50)
    print(f" RDD ANALYSIS FOR: {col.upper()} ")
    print("="*50)
    
    # ---------------------------------------------------------
    # MODEL 1: Discontinuity in VALUE only (Constant Slope)
    # Formula: value ~ time_centered + post_event
    # ---------------------------------------------------------
    model1 = smf.ols(formula=f'{col} ~ time_centered + post_event', data=df).fit()
    
    print("--- MODEL 1: Intercept Shift Only ---")
    print(f"Jump Estimate (post_event Coeff): {model1.params['post_event']:.4f}")
    print(f"P-value for Jump: {model1.pvalues['post_event']:.4e}")
    print(f"Model R-squared: {model1.rsquared:.4f}\n")
    
    # ---------------------------------------------------------
    # MODEL 2: Discontinuity in VALUE and SLOPE (Derivative)
    # Formula: value ~ time_centered + post_event + interaction
    # ---------------------------------------------------------
    model2 = smf.ols(formula=f'{col} ~ time_centered + post_event + interaction', data=df).fit()
    
    print("--- MODEL 2: Intercept and Slope Shift ---")
    print(f"Jump Estimate (post_event Coeff): {model2.params['post_event']:.4f}")
    print(f"P-value for Jump: {model2.pvalues['post_event']:.4e}")
    print(f"Slope Change (interaction Coeff): {model2.params['interaction']:.4f}")
    print(f"P-value for Slope Change: {model2.pvalues['interaction']:.4e}")
    print(f"Model R-squared: {model2.rsquared:.4f}")



 RDD ANALYSIS FOR: VALUE1 
--- MODEL 1: Intercept Shift Only ---
Jump Estimate (post_event Coeff): 0.8508
P-value for Jump: 8.5607e-02
Model R-squared: 0.6513

--- MODEL 2: Intercept and Slope Shift ---
Jump Estimate (post_event Coeff): 0.9035
P-value for Jump: 2.0170e-02
Slope Change (interaction Coeff): 0.1053
P-value for Slope Change: 3.6021e-12
Model R-squared: 0.7897

 RDD ANALYSIS FOR: VALUE2 
--- MODEL 1: Intercept Shift Only ---
Jump Estimate (post_event Coeff): 0.6827
P-value for Jump: 1.1310e-01
Model R-squared: 0.5399

--- MODEL 2: Intercept and Slope Shift ---
Jump Estimate (post_event Coeff): 0.7012
P-value for Jump: 9.4565e-02
Slope Change (interaction Coeff): 0.0369
P-value for Slope Change: 1.1811e-02
Model R-squared: 0.5695

 RDD ANALYSIS FOR: VALUE3 
--- MODEL 1: Intercept Shift Only ---
Jump Estimate (post_event Coeff): 1.7673
P-value for Jump: 3.2793e-05
Model R-squared: 0.7773

--- MODEL 2: Intercept and Slope Shift ---
Jump Estimate (post_event Coeff): 1.7926
P-v

Q1. Which dataset is most likely to have a discontinuity (or has the strongest discontinuity) in the value at the event (at time = 50)? 


A1. value3

Q2. Which dataset is most likely to have a discontinuity (or has the strongest discontinuity) in the derivative (at time = 50)? 

A2. value1